# Spark Structured Streaming with Kafka

This notebook consumes live equity-market data from a Kafka cluster using **Spark Structured Streaming**. A separate producer process (`producer.py`) polls Yahoo Finance and publishes to two topics:

| Topic | Cadence | Payload |
|---|---|---|
| `trades` | every ~5 s | `symbol`, `price`, `volume`, `timestamp` |
| `news`   | every ~minute, deduplicated | `symbol`, `headline`, `link`, `published`, `timestamp` |

We will use them to demonstrate the standard streaming patterns:

1. [Setup](#1.-Setup)
2. [Reading the trades stream from Kafka](#2.-Reading-the-trades-stream)
3. [Sinks and output modes](#3.-Sinks-and-output-modes)
4. [Windowed aggregations and watermarks](#4.-Windowed-aggregations)
5. [Stream-stream join: news → trade reaction](#5.-Stream-stream-join)
6. [Stream-static join: enrich trades with sector / market cap](#6.-Stream-static-join)
7. [Bonus: simulating a stream from a CSV directory](#7.-Simulating-a-stream-from-files)

Structured Streaming treats a stream as a table that updates over time, so the same DataFrame API works on both bounded and unbounded data.

## 1. Setup

We import Spark, declare the schema types we'll need, and start a local Spark session with the Kafka connector on the classpath.

In [1]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, LongType, TimestampType,
)

KAFKA_BOOTSTRAP_SERVERS = "localhost:8098"

In [ ]:
spark = (
    SparkSession.builder
    .appName("kafka_streaming")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.shuffle.partitions", 4)
    .master("local[*]")
    .getOrCreate()
)

## 2. Reading the trades stream

We subscribe to the `trades` topic, declare its JSON schema, and parse the Kafka `value` column into a typed DataFrame. The result, `trades_df`, is the canonical streaming DataFrame we'll reuse throughout the rest of the notebook.

In [3]:
trade_schema = StructType([
    StructField("symbol",    StringType()),
    StructField("price",     DoubleType()),
    StructField("volume",    LongType()),
    StructField("timestamp", TimestampType()),
])

raw_trades = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", "trades")
    .option("startingOffsets", "earliest")
    .load()
)

# Parse the JSON value into a typed DataFrame. 
trades_df = (
    raw_trades
    .select(F.from_json(F.col("value").cast("string"), trade_schema).alias("v"))
    .select(
        F.col("v.symbol").alias("symbol"),
        F.col("v.price").alias("price"),
        F.col("v.volume").alias("volume"),
        F.col("v.timestamp").alias("event_time"),
    )
    .filter(F.col("price") > 0)
)

Like the batch API, streaming operations are lazy — nothing happens until we attach a sink and call `.start()`.

## 3. Sinks and output modes

Structured Streaming supports several sinks: **Kafka**, **file** (parquet/JSON/CSV), **foreach** for custom logic, **console** for debugging, and **memory** for interactive inspection from SQL.

Each query also has an *output mode* that controls what gets written on each trigger:

- `append` — only new rows since the last trigger
- `update` — only rows whose aggregate value changed
- `complete` — the entire result table (only valid for aggregates)

We'll use the **memory** sink with a query name, which lets us run ad-hoc Spark SQL against the rolling stream. Run the next cell, wait a few seconds for trades to flow, then run the SQL query.

In [ ]:
trades_query = (
    trades_df.writeStream
    .outputMode("append")
    .queryName("trades_table")
    .format("memory")
    .start()
)

In [ ]:
spark.sql("""
    SELECT symbol,
           COUNT(*)    AS tick_count,
           AVG(price)  AS avg_price,
           MAX(volume) AS session_volume
    FROM trades_table
    GROUP BY symbol
    ORDER BY tick_count DESC
""").show()

## 4. Windowed aggregations

Time-windowed aggregations over a tick stream are an important part of streaming analytics.

A **watermark** tells the engine how late an event can arrive and still be included in its window. State for windows older than `max(event_time) - watermark` is dropped, which is what bounds the memory footprint of a long-running query.

In [ ]:
windowed = (
    trades_df
    .withWatermark("event_time", "30 seconds")
    .groupBy(
        F.window("event_time", "3 minutes"),
        "symbol",
    )
    .agg(
        F.count("*").alias("tick_count"),
        F.avg("price").alias("avg_price"),
        F.min("price").alias("low"),
        F.max("price").alias("high"),
        F.max("volume").alias("end_volume"),
    )
)

windowed_query = (
    windowed.writeStream
    .outputMode("complete")
    .queryName("trades_aggregated")
    .format("memory")
    .start()
)

In [ ]:
spark.sql("SELECT * FROM trades_aggregated ORDER BY window").show(truncate=False)

## 5. Stream-stream join

We now join the **`news`** stream with the **`trades`** stream to answer: *for each headline, what does the price do in the next 2 minutes after we observe it?* This is the canonical event-study join — the same shape behind news-momentum strategies and post-earnings drift research.

Three design points worth flagging:

1. **Event time = observation time, not `published`.** Yahoo's RSS lags reality — by the time the producer sees a headline, its `published` field may be hours old, well outside any meaningful trade window. We use the producer's observation `timestamp` as the event time so both streams share a real-time wall clock.
2. **Watermarks are tight.** Since both streams' event times are real-time observations, events are never more than seconds late. Tight watermarks (1 min for news, 30 s for trades) bound the join state and let Spark emit completed windows quickly.
3. **Interval condition.** The join condition uses a `BETWEEN` on the trade timestamp, which is what makes this a *time-bounded* stream-stream join with finite state.

In [10]:
news_schema = StructType([
    StructField("symbol",    StringType()),
    StructField("headline",  StringType()),
    StructField("link",      StringType()),
    StructField("published", TimestampType()),
    StructField("timestamp", TimestampType()),
])

raw_news = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", "news")
    .option("startingOffsets", "earliest")
    .load()
)

# Note: event_time is the producer's *observation* timestamp,
# not the article's published time. RSS publication is too low-frequency
# to align with the live trade stream.
news_df = (
    raw_news
    .select(F.from_json(F.col("value").cast("string"), news_schema).alias("v"))
    .select(
        F.col("v.symbol").alias("symbol"),
        F.col("v.headline").alias("headline"),
        F.col("v.published").alias("published"),
        F.col("v.timestamp").alias("event_time"),
    )
)

In [ ]:
# Watermark each side. Since `event_time` is the producer's
# observation timestamp (always 'now') for both streams, events
# are never more than a few seconds late - so the watermarks can
# be tight. 
news_wm   = news_df.withWatermark("event_time", "1 minute").alias("n")
trades_wm = trades_df.withWatermark("event_time", "30 seconds").alias("t")

# For each headline, find the trades on the same symbol that
# fired within the next 2 minutes. A shorter window means rows
# emit sooner.
joined = news_wm.join(
    trades_wm,
    F.expr("""
        n.symbol = t.symbol AND
        t.event_time BETWEEN n.event_time AND n.event_time + interval 2 minutes
    """),
)

# Roll up to one row per headline.
reaction = (
    joined.groupBy("n.symbol", "n.event_time", "n.headline")
    .agg(
        F.count("*").alias("trades_in_window"),
        F.first("t.price").alias("first_price"),
        F.last("t.price").alias("last_price"),
        F.min("t.price").alias("low"),
        F.max("t.price").alias("high"),
    )
)

reaction_query = (
    reaction.writeStream
    .outputMode("append")
    .queryName("news_reaction")
    .format("memory")
    .start()
)

**Expect the first rows to appear ~3-4 minutes after starting the producer.** Stream-stream joins only support `append` output mode, which means Spark holds each window's output until the watermark has passed the window's end. With a 2-minute join window and a 1-minute news watermark, the earliest emission is ~3 minutes after the first news event arrives.

You may insert a ticker that interests you to subset on it, such as:     

```WHERE symbol LIKE 'MSFT'```


In [ ]:
spark.sql("""
    SELECT symbol,
           event_time,
           headline,
           trades_in_window,
           ROUND(((last_price - first_price) / first_price) * 100, 3) AS pct_move,
           low, high
    FROM news_reaction
    ORDER BY event_time DESC
""").show(truncate=False)

## 6. Stream-static join

A stream can also be joined with a static DataFrame — typically a slowly-changing reference table. We'll enrich the trades stream with S&P 500 constituent metadata (sector, founding year) and then aggregate to compute the **average firm age per sector** weighted by trade activity.

We will use the `sp500_constituents.csv` provided on Moodle, which is extracted from Wikipedia.

In [22]:
constituents_schema = StructType([
    StructField("symbol",  StringType()),
    StructField("name",    StringType()),
    StructField("sector",  StringType()),
    StructField("founded", IntegerType()),
])

constituents_df = (
    spark.read.format("csv")
    .option("header", True)
    .schema(constituents_schema)
    .load("sp500_constituents.csv")
)

In [ ]:
# Enrich each trade with its constituent metadata, including the
# firm's age in years (computed from founding year).
enriched = (
    trades_df.join(constituents_df, on="symbol", how="inner")
    .withColumn("firm_age", F.year(F.current_timestamp()) - F.col("founded"))
    .select("event_time", "symbol", "name", "sector",
            "price", "volume", "founded", "firm_age")
)

enriched_query = (
    enriched.writeStream
    .queryName("trades_sector_info")
    .outputMode("append")
    .format("memory")
    .start()
)

In [ ]:
# Average firm age per sector
# Here, we use an SQL context expression
spark.sql("""
    WITH firms AS (
        SELECT DISTINCT symbol, sector, firm_age
        FROM trades_sector_info
        WHERE firm_age IS NOT NULL
    )
    SELECT sector,
           COUNT(*)                AS n_firms,
           ROUND(AVG(firm_age), 1) AS avg_firm_age
    FROM firms
    GROUP BY sector
    ORDER BY avg_firm_age DESC
""").show(truncate=False)

## 7. Simulating a stream from files

Spark can also treat a directory of files as a stream: each new file dropped into the directory becomes a micro-batch. This is useful for testing pipelines without spinning up Kafka, or for replaying historical data through the same code path used in production.

We will populate `ohlc/` with one CSV per trading day, then point a `readStream` at the directory. Spark picks files up in name order, which is why we use ISO dates (`YYYY-MM-DD.csv`) as filenames.

In [ ]:
import os
import polars as pl
import yfinance as yf

OHLC_DIR = "ohlc"
OHLC_SYMBOLS = ["AAPL", "MSFT", "NVDA", "GOOGL", "META",
                "AMZN", "TSLA", "JPM", "XOM", "JNJ"]

os.makedirs(OHLC_DIR, exist_ok=True)

# Pull 60 days of daily bars per symbol. yfinance only returns
# pandas, so we convert immediately and stay in polars from there.
frames = []
for sym in OHLC_SYMBOLS:
    pdf = yf.Ticker(sym).history(period="60d", auto_adjust=False)
    df = (
        pl.from_pandas(pdf.reset_index())
        .rename({"Date": "timestamp", "Open": "open", "High": "high",
                 "Low": "low", "Close": "close", "Volume": "volume"})
        .with_columns(symbol=pl.lit(sym))
        .select("timestamp", "symbol", "open", "high", "low", "close", "volume")
    )
    frames.append(df)

ohlc = pl.concat(frames)

# Write one file per trading day. ISO date filenames sort
# chronologically, which is the order Spark will pick them up in.
for (date,), group in ohlc.group_by(pl.col("timestamp").dt.date()):
    group.write_csv(f"{OHLC_DIR}/{date}.csv")

print(f"Wrote {len(os.listdir(OHLC_DIR))} files to {OHLC_DIR}/")

Now read the directory as a stream. The two key options are:

- `maxFilesPerTrigger` — how many new files to pick up per batch
- `trigger(processingTime=...)` — how often the engine wakes to look for new files

These together throttle the replay speed. With 3 files per 10-second trigger, the 60-day backfill takes ~3-4 minutes.

In [ ]:
ohlc_schema = StructType([
    StructField("timestamp", TimestampType()),
    StructField("symbol",    StringType()),
    StructField("open",      DoubleType()),
    StructField("high",      DoubleType()),
    StructField("low",       DoubleType()),
    StructField("close",     DoubleType()),
    StructField("volume",    LongType()),
])

ohlc_stream = (
    spark.readStream
    .option("maxFilesPerTrigger", 3)
    .option("header", True)
    .format("csv")
    .schema(ohlc_schema)
    .load("ohlc")
)

# `processingTime` controls how often the engine wakes to look
# for new files; `maxFilesPerTrigger` caps files per batch.
ohlc_query = (
    ohlc_stream.writeStream
    .trigger(processingTime="10 seconds")
    .outputMode("append")
    .queryName("ohlc")
    .format("memory")
    .start()
)

In [ ]:
spark.sql("""
    SELECT symbol, timestamp, open, high, low, close, volume
    FROM ohlc
    ORDER BY timestamp DESC
""").show()

### Stopping the queries

Each `start()` returns a `StreamingQuery` handle. Stop them in reverse order before shutting the kernel:

In [ ]:
for q in [ohlc_query, enriched_query, reaction_query,
          windowed_query, trades_query]:
    q.stop()